In [12]:
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
import sys
sys.path

['/users/9/chen7751/miniconda3/envs/myenv2/lib/python38.zip',
 '/users/9/chen7751/miniconda3/envs/myenv2/lib/python3.8',
 '/users/9/chen7751/miniconda3/envs/myenv2/lib/python3.8/lib-dynload',
 '',
 '/users/9/chen7751/.local/lib/python3.8/site-packages',
 '/users/9/chen7751/miniconda3/envs/myenv2/lib/python3.8/site-packages',
 '/users/9/chen7751/miniconda3/envs/myenv2/lib/python3.8/site-packages/setuptools/_vendor',
 '/tmp/tmpxgxsf1b3']

In [14]:
from modeling_nmf import NMF, NMFConfig
from data_utils import get_reddit_data
from collections import defaultdict
import torch
from transformers import AutoTokenizer
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
import os

In [15]:
from tqdm import tqdm

def build_datastore_embedding(sentences, model, tokenizer, device):
    print('building datastore embeddings')
    model.eval()
    
    # Initialize an empty list to hold the sentence embeddings
    all_sentence_embeddings = []
    
    # Process sentences in batches
    batch_size = 64
    
    for i in tqdm(range(0, len(sentences), batch_size), total=len(sentences)//batch_size):
        batch_sentences = sentences[i:i+batch_size]
        
        # Tokenize sentences
        encoded_input = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors='pt').to(device)
        
        # Compute token embeddings
        with torch.no_grad():
            model_output = model(**encoded_input, output_hidden_states=True)
    
        # # Perform pooling
        # sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask']).cpu()
    
        # insteading of doing pooling, can just use last hidden state
        sentence_embeddings = model_output.hidden_states[-1][:,0,:]
        
        all_sentence_embeddings.append(sentence_embeddings.cpu())
    
    # Concatenate all batched embeddings
    all_sentence_embeddings = torch.cat(all_sentence_embeddings, dim=0)
    
    all_sentence_embeddings = F.normalize(all_sentence_embeddings, p=2, dim=1).cpu()
    post_embeddings = all_sentence_embeddings

    return post_embeddings

In [16]:
class KNNLMForTuningHyperParams:

    def __init__(
        self, 
        data_path,
        tokenizer_name_or_path = 'sentence-transformers/all-MiniLM-L6-v2',
        model_name_or_path = 'saved_models'
    ):


        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sentence_embedding_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        nmf = NMF.from_pretrained(model_name_or_path).to(device)
        nmf.eval()
        sentence_embedding_model = nmf.model
        
        training_dataset, validation_dataset, movie_vocab = get_reddit_data_with_heldout(data_path, heldout_portion=0.2)
        movie_vocab = [m.split(' (')[0] for m in movie_vocab] # remove the year at end
        training_posts = sorted(list(set(training_dataset['context'])))
        trainingpost2idx = dict(zip(
            training_posts, 
            list(range(len(training_posts)))
        ))

        trainingpostidx2movies = defaultdict(list)
        movie_from_posts = []
        for i in range(len(training_dataset['context'])):
            post_idx = trainingpost2idx[training_dataset['context'][i]]
            # the split is for removing year at end, e.g. " (2019)"
            # movie = movie_vocab[training_dataset['label'][i]].split(' (')[0] 
            trainingpostidx2movies[post_idx].append(training_dataset['label'][i])

        post_embeddings = build_datastore_embedding(
            sentences = training_posts, 
            model = sentence_embedding_model, 
            tokenizer = sentence_embedding_tokenizer, 
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        ).to(device)

        self.device = device
        self.sentence_embedding_tokenizer = sentence_embedding_tokenizer
        self.sentence_embedding_model = sentence_embedding_model
        self.movie_vocab = movie_vocab
        self.training_posts = training_posts
        self.post_embeddings = post_embeddings
        self.trainingpostidx2movies = trainingpostidx2movies
        self.nmf = nmf
        self.validation_dataset = validation_dataset

    def encode_sentences(self, batch_sentences):
        with torch.no_grad():
            encoded_input = self.sentence_embedding_tokenizer(
                batch_sentences, 
                padding=True, 
                truncation=True, 
                return_tensors='pt'
            ).to(self.device)
            with torch.no_grad():
                model_output = self.sentence_embedding_model(**encoded_input, output_hidden_states=True)
            sentence_embeddings = model_output.hidden_states[-1][:,0,:]
    
        return sentence_embeddings

    def top_post_ids_retrieval(self, query):
    
        query_embedding = self.encode_sentences(query)[0].reshape(1, -1)
        cosine_similarities = torch.cosine_similarity(query_embedding, self.post_embeddings).cpu().numpy()
        sorted_indices = np.argsort(cosine_similarities)[::-1]
        return sorted_indices, cosine_similarities[sorted_indices]

    def count_based_probability(
        self,
        query, 
        num_posts_to_consider=30, 
        return_logits=False, 
        distance_weighting=False,
        temperature = 1.0
    ):
        relevant_post_ids, similarities = self.top_post_ids_retrieval(query)
        movie_pool = defaultdict(int)
        probas = np.zeros(len(self.movie_vocab))
        for i, id in enumerate(relevant_post_ids[:num_posts_to_consider]):
            for movieid in self.trainingpostidx2movies[id]:
                if distance_weighting:
                    probas[movieid] += similarities[i]/temperature
                else:
                    probas[movieid] += 1
        if return_logits:
            return probas
        probas = torch.nn.functional.softmax(torch.from_numpy(probas), dim=-1)
        return probas

    def predictor_probability(
        self, 
        query,
        return_logits = False
    ):
        
        with torch.no_grad():
            model_input = self.sentence_embedding_tokenizer(
                query, 
                return_tensors='pt', 
                max_length=368, 
                truncation=True
            )

            logits = reddit_knnlm_recommender.nmf(
                            model_input['input_ids'].to(self.device), 
                            model_input['token_type_ids'].to(self.device), 
                            model_input['attention_mask'].to(self.device),
                            labels=None
                        ).logits
        if return_logits:
            return logits.cpu().numpy()
        probas = F.softmax(logits, dim=-1)[0]
        return  probas.cpu().numpy()

In [17]:
class KNNLMRecommender:

    def __init__(
        self, 
        tokenizer_name_or_path = 'sentence-transformers/all-MiniLM-L6-v2',
        model_name_or_path = 'saved_models',
        data_path = 'reddit/reddit_large_train.csv'
    ):


        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sentence_embedding_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        nmf = NMF.from_pretrained(model_name_or_path).to(device)
        nmf.eval()
        sentence_embedding_model = nmf.model
        
        training_dataset, movie_vocab = get_reddit_data(data_path)
        movie_vocab = [m.split(' (')[0] for m in movie_vocab] # remove the year at end
        training_posts = sorted(list(set(training_dataset['context'])))
        trainingpost2idx = dict(zip(
            training_posts, 
            list(range(len(training_posts)))
        ))

        trainingpostidx2movies = defaultdict(list)
        movie_from_posts = []
        for i in range(len(training_dataset['context'])):
            post_idx = trainingpost2idx[training_dataset['context'][i]]
            # the split is for removing year at end, e.g. " (2019)"
            # movie = movie_vocab[training_dataset['label'][i]].split(' (')[0] 
            trainingpostidx2movies[post_idx].append(training_dataset['label'][i])

        post_embeddings = build_datastore_embedding(
            sentences = training_posts, 
            model = sentence_embedding_model, 
            tokenizer = sentence_embedding_tokenizer, 
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )

        self.device = device
        self.sentence_embedding_tokenizer = sentence_embedding_tokenizer
        self.sentence_embedding_model = sentence_embedding_model
        self.movie_vocab = movie_vocab
        self.training_posts = training_posts
        self.post_embeddings = post_embeddings
        self.trainingpostidx2movies = trainingpostidx2movies
        self.nmf = nmf

    def encode_sentences(self, batch_sentences):
        with torch.no_grad():
            encoded_input = self.sentence_embedding_tokenizer(
                batch_sentences, 
                padding=True, 
                truncation=True, 
                return_tensors='pt'
            ).to(self.device)
            with torch.no_grad():
                model_output = self.sentence_embedding_model(**encoded_input, output_hidden_states=True)
            sentence_embeddings = model_output.hidden_states[-1][:,0,:]
    
        return sentence_embeddings

    def top_post_ids_retrieval(self, query):
    
        query_embedding = self.encode_sentences(query)[0].reshape(1, -1).cpu()
        cosine_similarities = cosine_similarity(query_embedding, self.post_embeddings)
        sorted_indices = np.argsort(cosine_similarities[0])[::-1]
        return sorted_indices, cosine_similarities[0][sorted_indices]

    def count_based_probability(
        self,
        query, 
        num_posts_to_consider=30, 
        return_logits=False, 
        distance_weighting=False
    ):
        relevant_post_ids, similarities = self.top_post_ids_retrieval(query)
        movie_pool = defaultdict(int)
        probas = np.zeros(len(self.movie_vocab))
        for i, id in enumerate(relevant_post_ids[:num_posts_to_consider]):
            for movieid in self.trainingpostidx2movies[id]:
                if distance_weighting:
                    probas[movieid] += similarities[i]
                else:
                    probas[movieid] += 1
        if return_logits:
            return probas
        probas = torch.nn.functional.softmax(torch.from_numpy(probas), dim=-1)
        return probas

    def predictor_probability(
        self, 
        query,
        return_logits = False
    ):
        
        with torch.no_grad():
            model_input = self.sentence_embedding_tokenizer(
                query, 
                return_tensors='pt', 
                max_length=368, 
                truncation=True
            )

            logits = reddit_knnlm_recommender.nmf(
                            model_input['input_ids'].to(self.device), 
                            model_input['token_type_ids'].to(self.device), 
                            model_input['attention_mask'].to(self.device),
                            labels=None
                        ).logits
        if return_logits:
            return logits.cpu().numpy()
        probas = F.softmax(logits, dim=-1)[0]
        return  probas.cpu().numpy()

In [18]:
import json

def get_eligible_entities(resource_path=''):
    entity2id = eval(open(resource_path+'entity2id.json', 'r').readlines()[0])
    id2entity = {v:k for k,v in entity2id.items()}
    eligible_entities = [id2entity[idx].split('/')[-1].split('_(')[0].rstrip('>').replace('_',' ') for idx in \
    eval(open(resource_path+'item_ids.json', 'r').readlines()[0])]
    return eligible_entities

In [19]:
inspired_eligible_entities = set(get_eligible_entities('./entity_assets/inspired/'))

In [20]:
redial_eligible_entities = set(get_eligible_entities('./entity_assets/redial/'))

## Inspired

In [21]:
from data_utils import get_reddit_data_with_heldout
import numpy as np
import pandas as pd
from scipy.stats import sem

# reddit_knnlm_recommender = KNNLMForTuningHyperParams(
#     data_path= 'datasets/inspired/inspired_train.csv',
#     model_name_or_path = 'models/inspired'
# )

# k=20
# n_neighbors = [15, 30, 60, 90, 120, 150, 180]
# recall = []
# for num_posts_to_consider in n_neighbors:
#     hits_at_k = []
#     cache = dict()
#     for idx in tqdm(range(len(reddit_knnlm_recommender.validation_dataset['context'])), total=len(reddit_knnlm_recommender.validation_dataset['context'])):
#         query = reddit_knnlm_recommender.validation_dataset['context'][idx]
#         target = reddit_knnlm_recommender.movie_vocab[reddit_knnlm_recommender.validation_dataset['label'][idx]]
#         if query not in cache:
#             movie_counts = reddit_knnlm_recommender.count_based_probability(
#                 query, 
#                 num_posts_to_consider=num_posts_to_consider, 
#                 return_logits=True
#             )
#             recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_counts)]
#             recommended_movies = [e for e in recommended_movies if e in inspired_eligible_entities]
#         else:
#             recommended_movies = cache[query]
        
#         hits_at_k.append(int(target in recommended_movies[:k]))
#     recall.append(np.mean(hits_at_k))


# best_n_neighbors = n_neighbors[np.argmax(recall)]

best_n_neighbors = 90 # comment it if you fine-tune this value
print('the recommended number of neighbors to use is ', best_n_neighbors)

testset = pd.read_csv('datasets/inspired/inspired_test.csv')
test_inputs = testset['test_inputs']
test_groundtruths = testset['test_outputs']

reddit_knnlm_recommender = KNNLMRecommender(
    data_path= 'datasets/inspired/inspired_train.csv',
    model_name_or_path = 'models/inspired'
)


K = [1,5, 10, 20,50,100,300]



the recommended number of neighbors to use is  90
interaction preservance rate at 20000 items
1.0
num items  1436
flattening posts into training data


100%|██████████| 731/731 [00:00<00:00, 1493441.90it/s]


building datastore embeddings


12it [00:01,  9.05it/s]                        


In [22]:
print('-------------------- Retrieval -------------------------------')
cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_counts = reddit_knnlm_recommender.count_based_probability(
                query, 
                num_posts_to_consider=best_n_neighbors, 
                return_logits=True
            )
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_counts)]
            recommended_movies = [e for e in recommended_movies if e in inspired_eligible_entities]
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = recommended_movies[:k].index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))
    
print('-------------------- Recommend -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]

    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_probas = reddit_knnlm_recommender.predictor_probability(
                query, 
                return_logits=False
            )
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_probas)]
            recommended_movies = [e for e in recommended_movies if e in inspired_eligible_entities]
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = recommended_movies[:k].index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))


# print('-------------------- R+R (rerank) -------------------------------')

# cache = dict()
# for k in K:
#     hits_at_k = []
#     for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
#         query = test_inputs[idx]
#         target = test_groundtruths[idx]
        
#         if query not in cache:
#             movie_counts = reddit_knnlm_recommender.count_based_probability(
#                 query, 
#                 num_posts_to_consider=best_n_neighbors, 
#                 return_logits=True
#             )
#             movie_probas = reddit_knnlm_recommender.predictor_probability(
#                 query, 
#                 return_logits=False
#             )
#             movie_scores = movie_counts + movie_probas
#             recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_scores)]
#             recommended_movies = [e for e in recommended_movies if e in inspired_eligible_entities]
#             cache[query] = recommended_movies
#         else:
#             recommended_movies = cache[query]
#         hits_at_k.append(int(target in recommended_movies[:k]))
#     print('r@'+str(k), np.mean(hits_at_k),'; se: ', sem(hits_at_k))

# print('-------------------- R+R (rerank) with small gamma (cleaner solution for paper) -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr= []
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_counts = reddit_knnlm_recommender.count_based_probability(
                query, 
                num_posts_to_consider=best_n_neighbors, 
                return_logits=False
            )
            movie_probas = reddit_knnlm_recommender.predictor_probability(
                query, 
                return_logits=False
            )
            movie_scores = movie_counts*(1-(1e-10)) + movie_probas*1e-10
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_scores)]
            recommended_movies = [e for e in recommended_movies if e in inspired_eligible_entities]
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = recommended_movies[:k].index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))

-------------------- Retrieval -------------------------------


100%|██████████| 211/211 [00:00<00:00, 218.56it/s]


r@: 1 0.037914691943127965 ; MRR@: 1 0.037914691943127965 ; se:  0.013179559946043735


100%|██████████| 211/211 [00:00<00:00, 205288.37it/s]


r@: 5 0.11374407582938388 ; MRR@: 5 0.06342812006319115 ; se:  0.021909593576363847


...

r@: 50 0.27014218009478674 ; MRR@: 50 0.07279844170657351 ; se:  0.030641194076293107


100%|██████████| 211/211 [00:00<00:00, 154180.86it/s]


r@: 100 0.32701421800947866 ; MRR@: 100 0.07351939995543995 ; se:  0.03237252797910216


100%|██████████| 211/211 [00:00<00:00, 97995.59it/s]

r@: 300 0.4549763033175355 ; MRR@: 300 0.07427508886521415 ; se:  0.03436310776018828


## Reddit

In [23]:
from data_utils import get_reddit_data_with_heldout
import numpy as np
import pandas as pd
from scipy.stats import sem

# reddit_knnlm_recommender = KNNLMForTuningHyperParams(
#     data_path= 'datasets/reddit/reddit_large_train.csv',
#     model_name_or_path = 'models/reddit'
# )

# k=20
# n_neighbors = [15, 30, 60, 90, 120, 150, 180]
# recall = []
# for num_posts_to_consider in n_neighbors:
#     hits_at_k = []
#     cache = dict()
#     for idx in tqdm(range(len(reddit_knnlm_recommender.validation_dataset['context'])), total=len(reddit_knnlm_recommender.validation_dataset['context'])):
#         query = reddit_knnlm_recommender.validation_dataset['context'][idx]
#         target = reddit_knnlm_recommender.movie_vocab[reddit_knnlm_recommender.validation_dataset['label'][idx]]
#         if query not in cache:
#             movie_counts = reddit_knnlm_recommender.count_based_probability(
#                 query, 
#                 num_posts_to_consider=num_posts_to_consider, 
#                 return_logits=True
#             )
#             recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_counts)]
#         else:
#             recommended_movies = cache[query]
        
#         hits_at_k.append(int(target in recommended_movies[:k]))
#     recall.append(np.mean(hits_at_k))


# best_n_neighbors = n_neighbors[np.argmax(recall)]
# print('the recommended number of neighbors to use is ', best_n_neighbors)

best_n_neighbors = 30 # uncomment above to tune n_neighbors

testset = pd.read_csv('datasets/reddit/reddit_test.csv')
test_inputs = testset['test_inputs']
test_groundtruths = testset['test_outputs']

reddit_knnlm_recommender = KNNLMRecommender(
    data_path= 'datasets/reddit/reddit_large_train.csv',
    model_name_or_path = 'models/reddit'
)

K = [1,5, 10, 20,50,100,300]

interaction preservance rate at 20000 items
0.9965472793625889
num items  20000
flattening posts into training data


100%|██████████| 39928/39928 [00:00<00:00, 460904.66it/s]


building datastore embeddings


621it [00:48, 12.79it/s]                         


In [24]:
print('-------------------- Retrieval -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_counts = reddit_knnlm_recommender.count_based_probability(
                query, 
                num_posts_to_consider=best_n_neighbors, 
                return_logits=True
            )
            recommended_movies = list(np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_counts)])
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        
        # MRR
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = list(recommended_movies[:k]).index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))


print('-------------------- Recommend -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_probas = reddit_knnlm_recommender.predictor_probability(
                query, 
                return_logits=False
            )
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_probas)]
            cache[query] = list(recommended_movies)
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = list(recommended_movies[:k]).index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))



# print('-------------------- R+R (rerank) -------------------------------')

# cache = dict()
# for k in K:
#     hits_at_k = []
#     for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
#         query = test_inputs[idx]
#         target = test_groundtruths[idx]
        
#         if query not in cache:
#             movie_counts = reddit_knnlm_recommender.count_based_probability(
#                 query, 
#                 num_posts_to_consider=best_n_neighbors, 
#                 return_logits=True
#             )
#             movie_probas = reddit_knnlm_recommender.predictor_probability(
#                 query, 
#                 return_logits=False
#             )
#             movie_scores = movie_counts + movie_probas
#             recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_scores)]
#             cache[query] = recommended_movies
#         else:
#             recommended_movies = cache[query]
#         hits_at_k.append(int(target in recommended_movies[:k]))
#     print('r@'+str(k), np.mean(hits_at_k),'; se: ', sem(hits_at_k))

print('-------------------- R+R (rerank) with small gamma (cleaner solution for paper) -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_counts = reddit_knnlm_recommender.count_based_probability(
                query, 
                num_posts_to_consider=best_n_neighbors, 
                return_logits=False
            )
            movie_probas = reddit_knnlm_recommender.predictor_probability(
                query, 
                return_logits=False
            )
            movie_scores = movie_counts*(1-(1e-10)) + movie_probas*1e-10
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_scores)]
            cache[query] = list(recommended_movies)
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = list(recommended_movies[:k]).index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))

-------------------- Retrieval -------------------------------


100%|██████████| 19438/19438 [01:53<00:00, 171.32it/s]


r@: 1 0.013684535446033543 ; MRR@: 1 0.013684535446033543 ; se:  0.0008333133383100948


100%|██████████| 19438/19438 [00:00<00:00, 212526.21it/s]


r@: 5 0.060963062043420106 ; MRR@: 5 0.029305998559522584 ; se:  0.0017161692138531269


...

r@: 50 0.25537606749665603 ; MRR@: 50 0.041613832736122965 ; se:  0.0031278360112411622


100%|██████████| 19438/19438 [00:00<00:00, 163359.65it/s]


r@: 100 0.3435024179442329 ; MRR@: 100 0.04286458716660775 ; se:  0.0034061741393183547


100%|██████████| 19438/19438 [00:00<00:00, 99376.99it/s]

r@: 300 0.4770552525980039 ; MRR@: 300 0.04368478136692386 ; se:  0.0035825942642339023


## Redial

In [25]:
from data_utils import get_reddit_data_with_heldout
import numpy as np
import pandas as pd
from scipy.stats import sem

# reddit_knnlm_recommender = KNNLMForTuningHyperParams(
#     data_path= 'datasets/redial/redial_train.csv',
#     model_name_or_path = 'models/redial'
# )

# k=20
# n_neighbors = [15, 30, 60, 90, 120, 150, 180]
# recall = []
# for num_posts_to_consider in n_neighbors:
#     hits_at_k = []
#     cache = dict()
#     for idx in tqdm(range(len(reddit_knnlm_recommender.validation_dataset['context'])), total=len(reddit_knnlm_recommender.validation_dataset['context'])):
#         query = reddit_knnlm_recommender.validation_dataset['context'][idx]
#         target = reddit_knnlm_recommender.movie_vocab[reddit_knnlm_recommender.validation_dataset['label'][idx]]
#         if query not in cache:
#             movie_counts = reddit_knnlm_recommender.count_based_probability(
#                 query, 
#                 num_posts_to_consider=num_posts_to_consider, 
#                 return_logits=True
#             )
#             recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_counts)]
#             recommended_movies = [e for e in recommended_movies if e in redial_eligible_entities]
#         else:
#             recommended_movies = cache[query]
        
#         hits_at_k.append(int(target in recommended_movies[:k]))
#     recall.append(np.mean(hits_at_k))


# best_n_neighbors = n_neighbors[np.argmax(recall)]
# print('the recommended number of neighbors to use is ', best_n_neighbors)

best_n_neighbors = 60

testset = pd.read_csv('datasets/redial/redial_test.csv')
test_inputs = testset['test_inputs']
test_groundtruths = testset['test_outputs']

reddit_knnlm_recommender = KNNLMRecommender(
    data_path= 'datasets/redial/redial_train.csv',
    model_name_or_path = 'models/redial'
)

K = [1,5, 10, 20,50,100,300]

interaction preservance rate at 20000 items
1.0
num items  5140
flattening posts into training data


100%|██████████| 8929/8929 [00:00<00:00, 1607819.53it/s]


building datastore embeddings


136it [00:08, 16.44it/s]                         


In [26]:
print('-------------------- Retrieval -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_counts = reddit_knnlm_recommender.count_based_probability(
                query, 
                num_posts_to_consider=best_n_neighbors, 
                return_logits=True
            )
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_counts)]
            recommended_movies = [e for e in recommended_movies if e in redial_eligible_entities]
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = recommended_movies[:k].index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))


print('-------------------- Recommend -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_probas = reddit_knnlm_recommender.predictor_probability(
                query, 
                return_logits=False
            )
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_probas)]
            recommended_movies = [e for e in recommended_movies if e in redial_eligible_entities]
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = recommended_movies[:k].index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))



# print('-------------------- R+R (rerank) -------------------------------')

# cache = dict()
# for k in K:
#     hits_at_k = []
#     for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
#         query = test_inputs[idx]
#         target = test_groundtruths[idx]
        
#         if query not in cache:
#             movie_counts = reddit_knnlm_recommender.count_based_probability(
#                 query, 
#                 num_posts_to_consider=best_n_neighbors, 
#                 return_logits=True
#             )
#             movie_probas = reddit_knnlm_recommender.predictor_probability(
#                 query, 
#                 return_logits=False
#             )
#             movie_scores = movie_counts + movie_probas
#             recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_scores)]
#             recommended_movies = [e for e in recommended_movies if e in redial_eligible_entities]
#             cache[query] = recommended_movies
#         else:
#             recommended_movies = cache[query]
#         hits_at_k.append(int(target in recommended_movies[:k]))
#     print('r@'+str(k), np.mean(hits_at_k),'; se: ', sem(hits_at_k))


print('-------------------- R+R (rerank) with small gamma (cleaner solution for paper) -------------------------------')

cache = dict()
for k in K:
    hits_at_k = []
    mrr=[]
    for idx in tqdm(range(len(test_inputs)), total = len(test_inputs) ):
        query = test_inputs[idx]
        target = test_groundtruths[idx]
        
        if query not in cache:
            movie_counts = reddit_knnlm_recommender.count_based_probability(
                query, 
                num_posts_to_consider=best_n_neighbors, 
                return_logits=False
            )
            movie_probas = reddit_knnlm_recommender.predictor_probability(
                query, 
                return_logits=False
            )
            movie_scores = movie_counts*(1-(1e-10)) + movie_probas*1e-10
            recommended_movies = np.array(reddit_knnlm_recommender.movie_vocab)[np.argsort(-movie_scores)]
            recommended_movies = [e for e in recommended_movies if e in redial_eligible_entities]
            cache[query] = recommended_movies
        else:
            recommended_movies = cache[query]
        hits_at_k.append(int(target in recommended_movies[:k]))
        target_rank = 0
        if target in recommended_movies[:k]:
            target_index = recommended_movies[:k].index(target)
#             print(f'the rank is {target_index} among {len(recommended_movies[:k])} items')
            target_rank = 1 / (target_index + 1)
        mrr.append(target_rank)
        
    print('r@: '+str(k), np.mean(hits_at_k), '; MRR@: '+str(k), np.mean(mrr),'; se: ', sem(hits_at_k))

-------------------- Retrieval -------------------------------


100%|██████████| 4288/4288 [00:46<00:00, 93.11it/s] 


r@: 1 0.01632462686567164 ; MRR@: 1 0.01632462686567164 ; se:  0.001935400234206339


100%|██████████| 4288/4288 [00:00<00:00, 119372.75it/s]


r@: 5 0.07789179104477612 ; MRR@: 5 0.037231809701492535 ; se:  0.004093172498109496


...

r@: 50 0.2954757462686567 ; MRR@: 50 0.051766527645007165 ; se:  0.00696838523120645


100%|██████████| 4288/4288 [00:00<00:00, 126198.47it/s]


r@: 100 0.37803171641791045 ; MRR@: 100 0.052895575376265845 ; se:  0.007405791771387271


100%|██████████| 4288/4288 [00:00<00:00, 65877.35it/s]

r@: 300 0.5293843283582089 ; MRR@: 300 0.05382744973557971 ; se:  0.0076232820990675245
